# Classical Machine Learning — Model Comparison

## 1. Driver-Based Cross-Validation Results

## Consolidated Classical ML Experiments — All Resolutions

In [8]:
import json
import shutil
import pandas as pd
from pathlib import Path
from datetime import datetime, timezone

ROOT = Path("..")
results_dir = ROOT / "results/classical/driver_based"

# Models and resolutions
models = {
    "Logistic Regression": "logistic_regression",
    "Linear SVM": "linear_svm",
    "Random Forest": "random_forest",
    "KNN": "knn"
}

resolutions = {
    "64x48": 1260,
    "128x96": 5940,
    "256x192": 25668
}

# Select the optimized SVM experiments
svm_files = {
    "64x48": "linear_svm_64x48_primal",
    "128x96": "linear_svm_128x96_primal",
    "256x192": "linear_svm_256x192"
}

report = {
    "report_title": "Classical ML — Driver-Based HOG Experiments",
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "dataset": {
        "training_images": 15732,
        "training_drivers": 18,
        "classes": 10,
        "feature_type": "HOG"
    },
    "evaluation": {
        "method": "StratifiedGroupKFold",
        "n_splits": 4,
        "shuffle": True,
        "random_state": 42,
        "grouping": "subject",
        "metrics": ["accuracy", "macro_f1"],
        "test_set_used": False
    },
    "experiments": {},
    "comparison": [],
    "ranking": [],
    "missing_metadata": []
}

for model_name, prefix in models.items():

    report["experiments"][model_name] = {}

    for resolution, n_features in resolutions.items():

        # Use optimized SVM results
        if model_name == "Linear SVM":
            filename = svm_files[resolution]
        else:
            filename = f"{prefix}_{resolution}"

        csv_path = results_dir / f"{filename}.csv"
        metadata_path = results_dir / f"{filename}_metadata.json"

        # Require actual saved results
        if not csv_path.exists():
            raise FileNotFoundError(
                f"Missing experiment results: {csv_path}"
            )

        # Load fold-level results
        df = pd.read_csv(csv_path)

        # Verify all four folds
        assert len(df) == 4, f"Incorrect number of folds: {filename}"
        assert sorted(df["fold"].tolist()) == [1, 2, 3, 4]
        assert (df["resolution"].astype(str) == resolution).all()

        # Load complete metadata when available
        metadata = None

        if metadata_path.exists():
            with open(metadata_path) as f:
                metadata = json.load(f)
        else:
            report["missing_metadata"].append(
                str(metadata_path)
            )

        # Verify feature count when metadata contains it
        if metadata is not None:
            dataset_info = metadata.get("dataset", {})

            recorded_features = dataset_info.get(
                "hog_features",
                metadata.get("hog_features")
            )

            if recorded_features is not None:
                assert int(recorded_features) == n_features, (
                    f"Feature mismatch: {filename}"
                )

        # Verify consistent optimized SVM parameters
        if model_name == "Linear SVM":

            if metadata is None:
                raise ValueError(
                    f"SVM metadata required: {metadata_path}"
                )

            svm_params = metadata["model_configuration"]["svm_parameters"]

            expected_params = {
                "C": 1.0,
                "dual": False,
                "max_iter": 1000,
                "tol": 0.01,
                "random_state": 42
            }

            for param, expected in expected_params.items():
                assert svm_params[param] == expected, (
                    f"SVM parameter mismatch: {filename}, {param}"
                )

        # Calculate summary statistics from saved scores
        summary = {
            "mean_accuracy": float(df["accuracy"].mean()),
            "std_accuracy": float(df["accuracy"].std(ddof=0)),
            "mean_macro_f1": float(df["macro_f1"].mean()),
            "std_macro_f1": float(df["macro_f1"].std(ddof=0))
        }

        # Retrieve recorded total runtime
        total_runtime = None

        if metadata is not None:

            timing = metadata.get("timing", {})

            total_runtime = timing.get(
                "total_execution_seconds"
            )

            # Support older metadata formats
            if total_runtime is None:
                total_runtime = metadata.get(
                    "notebook_elapsed_seconds_approx",
                    metadata.get(
                        "total_execution_time_seconds_approx"
                    )
                )

        # Add runtime to comparison
        summary["total_runtime_seconds"] = total_runtime

        # Preserve full fold results and metadata
        report["experiments"][model_name][resolution] = {
            "resolution": resolution,
            "hog_features": n_features,
            "summary": summary,
            "fold_results": json.loads(
                df.to_json(orient="records", double_precision=15)
            ),
            "metadata": metadata,
            "source_csv": str(csv_path),
            "source_metadata": (
                str(metadata_path) if metadata is not None else None
            )
        }

        # Add summary to overall comparison
        report["comparison"].append({
            "model": model_name,
            "resolution": resolution,
            **summary
        })

# Rank all experiments by Macro-F1
report["ranking"] = sorted(
    report["comparison"],
    key=lambda x: (
        x["mean_macro_f1"],
        x["mean_accuracy"]
    ),
    reverse=True
)

report["best_experiment"] = report["ranking"][0]

# Preserve previous report before updating
output_path = results_dir / "classical_ml_all_models_report.json"

backup_path = results_dir / "classical_ml_all_models_report_backup.json"

if output_path.exists() and not backup_path.exists():
    shutil.copy2(output_path, backup_path)

# Save complete consolidated report
with open(output_path, "w") as f:
    json.dump(report, f, indent=4, allow_nan=False)

# Display summary
comparison_df = pd.DataFrame(report["comparison"])

comparison_df["accuracy_%"] = (
    comparison_df["mean_accuracy"] * 100
)

comparison_df["macro_f1_%"] = (
    comparison_df["mean_macro_f1"] * 100
)

print(
    comparison_df[
        ["model", "resolution", "accuracy_%",
         "macro_f1_%", "total_runtime_seconds"]
    ].round(2).to_string(index=False)
)

print("\nTotal experiments:", len(report["comparison"]))
print("Best experiment:", report["best_experiment"]["model"])
print("Best resolution:", report["best_experiment"]["resolution"])
print("\nConsolidated report saved:", output_path)

if report["missing_metadata"]:
    print("\nWARNING: Some experiments have no metadata JSON:")
    for path in report["missing_metadata"]:
        print("-", path)

              model resolution  accuracy_%  macro_f1_%  total_runtime_seconds
Logistic Regression      64x48       48.96       49.70                  29.80
Logistic Regression     128x96       61.05       61.09                  16.30
Logistic Regression    256x192       58.91       58.67                  19.73
         Linear SVM      64x48       43.45       42.97                  40.33
         Linear SVM     128x96       50.12       49.34                 157.44
         Linear SVM    256x192       55.20       54.35                 567.02
      Random Forest      64x48       52.94       51.35                  17.10
      Random Forest     128x96       55.54       53.91                  37.70
      Random Forest    256x192       53.26       51.46                  81.88
                KNN      64x48       44.59       43.60                   1.30
                KNN     128x96       53.39       51.70                   6.31
                KNN    256x192       53.07       51.00          

In [9]:
import json
from pathlib import Path

# 1. Load consolidated report
report_path = Path(
    "../results/classical/driver_based/classical_ml_all_models_report.json"
)

with open(report_path) as f:
    report = json.load(f)

# 2. Verify all 12 experiments are included
assert len(report["comparison"]) == 12, "Missing experiments!"

# 3. Verify no metadata files are missing
assert not report["missing_metadata"], "Missing metadata!"

# 4. Verify each experiment has results and metadata
for model, experiments in report["experiments"].items():
    for resolution, experiment in experiments.items():

        assert len(experiment["fold_results"]) == 4
        assert experiment["metadata"] is not None
        assert experiment["summary"]["total_runtime_seconds"] is not None

        print(f"✓ {model} | {resolution} | Verified")

# 5. Display best experiment
best = report["best_experiment"]

print("\nBest Model:", best["model"])
print("Best Resolution:", best["resolution"])
print(f"Accuracy: {best['mean_accuracy']:.2%}")
print(f"Macro-F1: {best['mean_macro_f1']:.2%}")

print("\nTotal experiments:", len(report["comparison"]))
print("Missing metadata:", report["missing_metadata"])

print("\n✅ All 12 classical ML experiments verified successfully!")

✓ Logistic Regression | 64x48 | Verified
✓ Logistic Regression | 128x96 | Verified
✓ Logistic Regression | 256x192 | Verified
✓ Linear SVM | 64x48 | Verified
✓ Linear SVM | 128x96 | Verified
✓ Linear SVM | 256x192 | Verified
✓ Random Forest | 64x48 | Verified
✓ Random Forest | 128x96 | Verified
✓ Random Forest | 256x192 | Verified
✓ KNN | 64x48 | Verified
✓ KNN | 128x96 | Verified
✓ KNN | 256x192 | Verified

Best Model: Logistic Regression
Best Resolution: 128x96
Accuracy: 61.05%
Macro-F1: 61.09%

Total experiments: 12
Missing metadata: []

✅ All 12 classical ML experiments verified successfully!
